In [4]:
import pandas as pd

df = pd.read_csv("../data/processed/tickets_with_priority.csv")

In [5]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

ticket_texts = df["ticket_text"].tolist()

ticket_embeddings = embedder.encode(
    ticket_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1495 [00:00<?, ?it/s]

In [6]:
import os
import faiss
import pickle

# Create vector_store directory if it doesn't exist
os.makedirs("../vector_store", exist_ok=True)

embeddings = ticket_embeddings.astype("float32")

dimension = embeddings.shape[1]

ticket_index = faiss.IndexFlatL2(dimension)
ticket_index.add(embeddings)

faiss.write_index(ticket_index, "../vector_store/ticket_faiss.index")

ticket_metadata = df[["ticket_text", "category", "priority"]].to_dict(orient="records")

with open("../vector_store/ticket_metadata.pkl", "wb") as f:
    pickle.dump(ticket_metadata, f)

In [7]:
def retrieve_similar_tickets(query, top_k=5):
    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    distances, indices = ticket_index.search(query_embedding, top_k)

    results = []

    for idx, dist in zip(indices[0], distances[0]):
        item = ticket_metadata[idx].copy()
        item["distance"] = float(dist)
        results.append(item)

    return results

retrieve_similar_tickets("User cannot access Microsoft 365 account after password reset.")

[{'ticket_text': 'cannot access account cannot access hello cannot currently access even if reset password does work can you help cheers front end consultant',
  'category': 'Hardware',
  'priority': 'Medium',
  'distance': 0.6918518543243408},
 {'ticket_text': 'password reset contacted saying he cannot log after resetting his he also stated helped him issue',
  'category': 'Access',
  'priority': 'Medium',
  'distance': 0.7059458494186401},
 {'ticket_text': 'password expired his password reset had expired',
  'category': 'Access',
  'priority': 'Medium',
  'distance': 0.7097371816635132},
 {'ticket_text': 'can access systems after password reset sent wednesday july problem hello colleague having problems signing other since change his password yesterday evening could you please take look into kind regards developer',
  'category': 'Access',
  'priority': 'Medium',
  'distance': 0.7223801612854004},
 {'ticket_text': 'password reset called said he locked out his pc unable change his',
 